# IBF Toy Model: The Complete Lifecycle in Two Dimensions

This notebook instantiates the simplest possible IBF system: **d=2 configuration space, k=2 actions, two sequential contexts** where the second context contradicts part of what was learned in the first.

Every mechanism from the formal theory (§3) is visible here at a scale the reader can keep in mind. The seven steps below correspond directly to **Sections 2.2–2.8** of the paper.

## What This Notebook Produces

1. **The full IBF run** (seed=42): two phases of 25 epochs × 200 training points each
2. **An 8-panel figure** showing each lifecycle stage as a 2D landscape
3. **A mini-ablation table** (Table in §2.9) isolating the contribution of each mechanism
4. **Detailed telemetry**: dissolution events, agency modulation, learning curves

## Environment Structure (Eq. 1)

The scoring function decomposes into two terms:

$$s_j(x, c) = \beta \cdot x_1 \cdot p_j + \alpha \cdot u_c \cdot x_2 \cdot r_j$$

- The **first term** ($x_1$) is **invariant**: same correct answer in both contexts.
- The **second term** ($x_2$) is **context-specific**: flips sign between Context A and B.

The system is never told which feature is which. It must discover that from interaction alone.

## How to Read the Code

The single code cell below contains 10 clearly labeled parts:

| Part | Role | Paper Section |
|------|------|---------------|
| 1 | Configuration & hyperparameters | §2.1 |
| 2 | **IBFAgent** — the universal engine | §4 (Read Path §4.2, Write Path §4.3, Lifecycle §4.4) |
| 3 | Environment, encoder, base evaluator | §2.1, §5 |
| 4 | Geometric sigma calibration | §5.1 |
| 5 | Evaluation | §6.3 |
| 6 | Training loop with snapshots | §2.2–2.8 |
| 7 | Run full model | — |
| 8 | 8-panel figure | Figures 1–8 |
| 9 | Mini-ablation table | §2.9 |
| 10 | Detailed telemetry & curves | — |

In [24]:
# ══════════════════════════════════════════════════════════════════════════════
#  IBF 2D TOY MODEL — Paper Figure 1 Generator
#  Uses the EXACT IBFEngine from Cell 6 (RRW v4.17)
#  Adapted: d=2, k=2, 2 contexts, 200 training points, 500 eval points
#
#  sigma NOTE: The toy model uses sigma = median_dist / sqrt(2*d_eff) ~ 0.8,
#  NOT GRP P10/3 ~ 0.2. At d=2 with N=200, P10/3 isolates every center and
#  prevents cross-context kernel activation. The wider kernel makes ALL
#  mechanisms visible. The three experimental domains use GRP-calibrated
#  sigma; the toy model uses a pedagogically appropriate bandwidth.
# ══════════════════════════════════════════════════════════════════════════════

import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import Normalize, TwoSlopeNorm
from matplotlib.lines import Line2D
from dataclasses import dataclass, field
from typing import List, Dict, Tuple
from collections import defaultdict
import copy, os, time

OUTPUT_DIR = "toy_model_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)
SEED = 42
np.random.seed(SEED)

print("IBF 2D Toy Model -- Figure 1 Generator")
print("="*60)

# ============================================================================
#  PART 1 -- CONFIGURATION
# ============================================================================

@dataclass
class Config:
    d: int = 2
    k: int = 2
    N_pool: int = 200
    E: int = 25
    N_test: int = 500
    eval_every: int = 50
    eta: float = 0.10
    eta_cryst: float = 0.005
    mu_base: float = 0.06
    mu_cryst: float = 0.001
    v_max: float = 0.50
    cryst_n_min: int = 15
    activation_thresh: float = 0.18
    creation_thresh: float = 0.30
    convergence_threshold: float = 0.075
    capacity: int = 2000
    alpha_shrink: float = 1.0
    sigma_floor: float = 0.15
    min_samples_shrink: int = 50
    k_0: float = 5.0
    k_min: float = 1.0
    eta_k: float = 0.05
    w_max: float = 5.0
    w_dvar_threshold: float = 0.045
    n_cross_min: int = 4
    reversal_threshold: float = -0.125
    action_sep_target: float = 3.5
    action_sep_margin: float = 4.0
    action_sep_max_iter: int = 5

C = Config()
ACTION_EMB = np.eye(C.k)

p_vec = np.array([+1.0, -1.0])
r_vec = np.array([+1.0, -1.0])

print("  d=%d, k=%d, N_pool=%d, E=%d, N_test=%d" % (C.d, C.k, C.N_pool, C.E, C.N_test))
print("  eta=%.2f, mu_base=%.2f, mu_cryst=%.3f" % (C.eta, C.mu_base, C.mu_cryst))
print("  n_cross_min=%d, rev_thresh=%.3f" % (C.n_cross_min, C.reversal_threshold))


# ============================================================================
#  PART 2 -- IBF ENGINE (Cell 6 logic, VERBATIM from RRW v4.17)
# ============================================================================

@dataclass
class MemoryCenter:
    z: np.ndarray
    v: float = 0.0
    w: float = 0.0
    n_updates: int = 0
    D_sum: float = 0.0
    D_sq_sum: float = 0.0
    mu_eff: float = 0.06
    context_id: int = -1
    birth_step: int = 0
    context_update_counts: Dict[int, int] = field(default_factory=lambda: defaultdict(int))
    sigma: float = 0.58
    D_history: List[float] = field(default_factory=list)
    D_step_history: List[int] = field(default_factory=list)
    xj_history: List[Tuple] = field(default_factory=list)
    was_ever_crystallized: bool = False
    crucible_verified: bool = False
    dissolution_log: List[Dict] = field(default_factory=list)

    def D_var(self):
        if self.n_updates < 20:
            return 0.0
        rec = self.D_history[20:][-50:]
        if len(rec) < 2:
            return 0.0
        return float(np.var(rec))

    def is_crystallized(self):
        return self.mu_eff < C.mu_base - 1e-6

    def n_cross_updates(self):
        return sum(cnt for ctx, cnt in self.context_update_counts.items()
                   if ctx != self.context_id)


class IBFAgent:
    def __init__(self, sigma, merge_threshold, encoder, base_eval,
                 enable_crystallization=True, enable_crucible=True,
                 enable_agency=True):
        self.centers = []
        self.sigma_base = sigma
        self.merge_thresh = merge_threshold
        self.encoder = encoder
        self.base_eval = base_eval
        self.current_context = -1
        self.global_step = 0
        self.enable_crystallization = enable_crystallization
        self.enable_crucible = enable_crucible
        self.enable_agency = enable_agency
        self._merge_ratio = merge_threshold / sigma
        self._dynamic_sigma_floor = min(C.sigma_floor, sigma * 0.25)
        self._needle_threshold = sigma * 0.50

    def set_context(self, ctx_id):
        if ctx_id != self.current_context:
            for c in self.centers:
                c.crucible_verified = False
        self.current_context = ctx_id

    def kernel_batch(self, z, centers=None):
        cs = centers if centers is not None else self.centers
        if not cs:
            return np.array([])
        Z = np.array([c.z for c in cs])
        sq = np.sum((Z - z[np.newaxis, :])**2, axis=1)
        sigmas = np.array([c.sigma for c in cs])
        return np.exp(-sq / (2.0 * sigmas**2))

    def _read_gate(self, c):
        if c.context_id == self.current_context:
            return 1.0
        if c.is_crystallized() and c.crucible_verified:
            return 1.0
        return 0.0

# ════════════════════════════════════════════
#  READ PATH (§4.2): Kernel readout, gating, action selection
# ════════════════════════════════════════════

    def delta_R(self, z):
        if not self.centers:
            return 0.0
        K = self.kernel_batch(z)
        total = 0.0
        for i, c in enumerate(self.centers):
            g = self._read_gate(c)
            if g > 0 and K[i] > C.activation_thresh:
                total += g * c.v * K[i]
        return total

    def delta_k(self, z):
        if not self.centers or not self.enable_agency:
            return 0.0
        K = self.kernel_batch(z)
        total_w, sum_K = 0.0, 0.0
        for i, c in enumerate(self.centers):
            if not c.is_crystallized():
                continue
            g = self._read_gate(c)
            if g > 0 and K[i] > C.activation_thresh:
                total_w += g * c.w * K[i]
                sum_K += g * K[i]
        return total_w / sum_K if sum_K > 1e-6 else 0.0

    def R_eff(self, z):
        return float(np.clip(self.base_eval(z) + self.delta_R(z), 0.0, 1.0))

    def R_eff_batch(self, Z_flat):
        R_base = self.base_eval.batch(Z_flat)
        if not self.centers:
            return np.clip(R_base, 0.0, 1.0)
        Z_c = np.array([c.z for c in self.centers])
        sigmas = np.array([c.sigma for c in self.centers])
        vs = np.array([c.v for c in self.centers])
        gate = np.array([1.0 if (c.context_id == self.current_context
                                  or (c.is_crystallized() and c.crucible_verified))
                         else 0.0 for c in self.centers])
        Z_sq = np.sum(Z_flat**2, axis=1, keepdims=True)
        C_sq = np.sum(Z_c**2, axis=1, keepdims=True)
        sq_d = Z_sq + C_sq.T - 2.0 * (Z_flat @ Z_c.T)
        K = np.exp(-sq_d / (2.0 * sigmas[np.newaxis, :]**2))
        K[K < C.activation_thresh] = 0.0
        return np.clip(R_base + K @ (gate * vs), 0.0, 1.0)

    def k_eff(self, z):
        return max(C.k_min, C.k_0 + self.delta_k(z))

    def select_action(self, z_all, deterministic=False):
        R = np.array([self.R_eff(z) for z in z_all])
        if deterministic:
            return int(np.argmax(R)), R, np.full(C.k, C.k_0), 0.0
        k = np.array([self.k_eff(z) for z in z_all])
        logits = k * R
        logits -= logits.max()
        probs = np.exp(logits) / np.sum(np.exp(logits))
        chosen = int(np.random.choice(C.k, p=probs))
        entropy = float(-np.sum(probs * np.log(probs + 1e-12)))
        return chosen, R, k, entropy

    def _thermodynamic_shrink(self, center):
        if center.n_updates >= C.min_samples_shrink:
            dvar = center.D_var()
            calc = max(self._dynamic_sigma_floor,
                       self.sigma_base / (1.0 + C.alpha_shrink * dvar))
            center.sigma = min(center.sigma, calc)

# ════════════════════════════════════════════
#  WRITE PATH (§4.3): Discrepancy-driven local modification
# ════════════════════════════════════════════

    def _update_agency(self, center, k_weight):
        if center.is_crystallized():
            dv = center.D_var()
            tw = np.clip(C.w_max * (1.0 - dv / C.w_dvar_threshold),
                         -C.w_max, C.w_max)
            center.w += C.eta_k * k_weight * (tw - center.w)
            center.w = np.clip(center.w, -C.w_max, C.w_max)

    def update(self, z_chosen, D, x=None, j_chosen=None):
        self.global_step += 1
        K_all = self.kernel_batch(z_chosen) if self.centers else np.array([])

        if self.enable_crucible:
            for i, c in enumerate(self.centers):
                if c.is_crystallized() and c.context_id != self.current_context:
                    kw = float(K_all[i])
                    if kw >= C.activation_thresh:
                        juris_D = D * kw
                        c.v = np.clip(c.v + C.eta_cryst * juris_D, -C.v_max, C.v_max)
                        c.n_updates += 1
                        c.D_sum += juris_D
                        c.D_sq_sum += juris_D * juris_D
                        c.context_update_counts[self.current_context] += 1
                        c.D_history.append(D)
                        c.D_step_history.append(self.global_step)
                        if x is not None:
                            c.xj_history.append((x.copy(), j_chosen))

        li = [i for i, c in enumerate(self.centers)
              if c.context_id == self.current_context]
        max_K = float(np.max(K_all[li])) if li else 0.0

        if max_K < C.creation_thresh:
            if len(self.centers) < C.capacity:
                juris_D = D * 1.0
                nc = MemoryCenter(
                    z=z_chosen.copy(),
                    v=np.clip(C.eta * juris_D, -C.v_max, C.v_max),
                    w=0.0, n_updates=1,
                    D_sum=juris_D, D_sq_sum=juris_D*juris_D,
                    mu_eff=C.mu_base,
                    context_id=self.current_context,
                    birth_step=self.global_step,
                    sigma=self.sigma_base)
                nc.context_update_counts[self.current_context] = 1
                nc.D_history.append(juris_D)
                nc.D_step_history.append(self.global_step)
                if x is not None:
                    nc.xj_history.append((x.copy(), j_chosen))
                self.centers.append(nc)
            return

        for i in li:
            c = self.centers[i]
            kw = float(K_all[i])
            if kw < C.activation_thresh:
                continue
            juris_D = D * kw
            effective_eta = C.eta_cryst if c.is_crystallized() else C.eta
            c.v = np.clip(c.v + effective_eta * juris_D, -C.v_max, C.v_max)
            c.n_updates += 1
            c.D_sum += juris_D
            c.D_sq_sum += juris_D * juris_D
            c.context_update_counts[self.current_context] += 1
            c.D_history.append(juris_D)
            c.D_step_history.append(self.global_step)
            if x is not None:
                c.xj_history.append((x.copy(), j_chosen))
            if self.enable_agency:
                self._update_agency(c, kw)
            self._thermodynamic_shrink(c)

    def end_epoch(self):
        for c in self.centers:
            c.v *= (1.0 - c.mu_eff)
            c.w *= (1.0 - c.mu_eff)

        for c in self.centers:
            if self.enable_crystallization:
                hist_len = len(c.D_history)
                cryst_grad = c.D_history[-50:] if hist_len > 0 else [0.0]
                cryst_mu = float(np.mean(cryst_grad))
                is_converged = abs(cryst_mu) < C.convergence_threshold

                if (not c.is_crystallized()
                        and c.n_updates >= C.cryst_n_min
                        and is_converged):
                    c.mu_eff = C.mu_cryst
                    c.was_ever_crystallized = True

                elif c.is_crystallized():
                    nc = c.n_cross_updates()
                    if nc >= C.n_cross_min:
                        crucible_grad = c.D_history[-C.n_cross_min:]
                        crucible_mu = float(np.mean(crucible_grad))
                        if (c.v * crucible_mu) < C.reversal_threshold:
                            c.dissolution_log.append({
                                'step': self.global_step,
                                'v': float(c.v),
                                'mu_D_recent': float(crucible_mu),
                                'product': float(c.v * crucible_mu),
                                'n_updates': c.n_updates,
                                'n_cross': nc,
                                'context': self.current_context,
                            })
                            c.mu_eff = C.mu_base
                            c.crucible_verified = False
                        else:
                            c.crucible_verified = True
            else:
                c.mu_eff = C.mu_base

        self._merge()

    def _merge(self):
        if len(self.centers) < 2:
            return
        merged = set()
        new = []
        for i in range(len(self.centers)):
            if i in merged:
                continue
            best = self.centers[i]
            for j in range(i+1, len(self.centers)):
                if j in merged:
                    continue
                if self.centers[i].context_id != self.centers[j].context_id:
                    continue
                dist = np.linalg.norm(self.centers[i].z - self.centers[j].z)
                ni = self.centers[i].sigma < self._needle_threshold
                nj = self.centers[j].sigma < self._needle_threshold
                if ni and nj:
                    th = self._merge_ratio * max(
                        self.centers[i].sigma, self.centers[j].sigma) * 1.5
                else:
                    th = self._merge_ratio * min(
                        self.centers[i].sigma, self.centers[j].sigma)
                if dist < th:
                    other = self.centers[j]
                    if other.n_updates > best.n_updates:
                        best, other = other, best
                    best.v = np.clip(best.v + other.v, -C.v_max, C.v_max)
                    best.w = np.clip(best.w + other.w, -C.w_max, C.w_max)
                    best.n_updates += other.n_updates
                    best.D_sum += other.D_sum
                    best.D_sq_sum += other.D_sq_sum
                    for ctx, cnt in other.context_update_counts.items():
                        best.context_update_counts[ctx] += cnt
                    best.D_history.extend(other.D_history)
                    best.D_step_history.extend(other.D_step_history)
                    best.xj_history.extend(other.xj_history)
                    best.sigma = min(best.sigma, other.sigma)
                    merged.add(j)
            new.append(best)
        if len(new) > C.capacity:
            cryst = [c for c in new if c.is_crystallized()]
            trans = sorted([c for c in new if not c.is_crystallized()],
                           key=lambda c: abs(c.v) * c.n_updates)
            keep = C.capacity - len(cryst)
            new = cryst + trans[-keep:] if keep > 0 else cryst[:C.capacity]
        self.centers = new

    def count_crystallized(self):
        return sum(1 for c in self.centers if c.is_crystallized())

    def count_verified(self):
        return sum(1 for c in self.centers
                   if c.is_crystallized() and c.crucible_verified)


# ============================================================================
#  PART 3 -- ENVIRONMENT, ENCODER, BASE EVALUATOR
# ============================================================================

class ToyEnvironment:
    def __init__(self, seed):
        rng = np.random.RandomState(seed)
        self.pool = rng.randn(C.N_pool, C.d)
        self.test_A = rng.randn(C.N_test, C.d)
        self.test_B = rng.randn(C.N_test, C.d)
        self.u = {'A': +1.0, 'B': -1.0}

    def score_clean(self, x, ctx):
        u_c = self.u[ctx]
        s = np.zeros(C.k)
        for j in range(C.k):
            s[j] = x[0] * p_vec[j] + u_c * x[1] * r_vec[j]
        return s

    def correct_action(self, x, ctx):
        return int(np.argmax(self.score_clean(x, ctx)))

    def correct_actions_batch(self, X, ctx):
        u_c = self.u[ctx]
        N = len(X)
        S = np.zeros((N, C.k))
        for j in range(C.k):
            S[:, j] = X[:, 0] * p_vec[j] + u_c * X[:, 1] * r_vec[j]
        return np.argmax(S, axis=1)

    def get_test(self, ctx):
        return self.test_A if ctx == 'A' else self.test_B


class ToyEncoder:
    def encode(self, x_np, action_idx):
        return np.concatenate([x_np, ACTION_EMB[action_idx]])
    def encode_batch(self, x_np, action_indices):
        return np.concatenate([x_np, ACTION_EMB[action_indices]], axis=1)


class ToyBaseEvaluator:
    def __init__(self, seed):
        rng = np.random.RandomState(seed)
        z_dim = C.d + C.k
        for var in [0.01, 0.02, 0.03, 0.05, 0.08, 0.10]:
            self.w = rng.randn(z_dim) * var
            self.b = 0.0
            spread = self._check_spread(rng)
            if spread >= 0.02:
                print("  Base evaluator spread: %.4f (variance=%.2f)" % (spread, var))
                return
        print("  Base evaluator spread: %.4f (max variance)" % spread)

    def _check_spread(self, rng):
        xs = rng.randn(250, C.d)
        acts = rng.randint(0, C.k, size=250)
        enc = ToyEncoder()
        Z = enc.encode_batch(xs, acts)
        spreads = []
        for i in range(0, 250 - C.k + 1, C.k):
            vals = self.batch(Z[i:i+C.k])
            spreads.append(vals.max() - vals.min())
        return float(np.mean(spreads))

    def __call__(self, z):
        return 1.0 / (1.0 + np.exp(-(np.dot(self.w, z) + self.b)))
    def batch(self, Z):
        return 1.0 / (1.0 + np.exp(-(Z @ self.w + self.b)))


class PassiveAgent:
    def __init__(self, encoder, base_eval):
        self.encoder = encoder
        self.base_eval = base_eval
    def R_eff_batch(self, Z_flat):
        return self.base_eval.batch(Z_flat)


# ============================================================================
#  PART 4 -- SIGMA CALIBRATION
#  Uses sigma = median_dist / sqrt(2*d_eff) ~ 0.8, NOT GRP P10/3 ~ 0.2
# ============================================================================

def calibrate_sigma(encoder, seed):
    rng = np.random.RandomState(seed + 999)
    xs = rng.randn(min(500, C.N_pool * 2), C.d)
    Z = encoder.encode_batch(xs, np.zeros(len(xs), dtype=int))
    Z_c = Z - Z.mean(axis=0)
    _, s, _ = np.linalg.svd(Z_c, full_matrices=False)
    var_exp = np.cumsum(s**2) / np.sum(s**2)
    eff_rank = int(np.searchsorted(var_exp, 0.95)) + 1
    dists = []
    for i in range(min(500, len(Z))):
        for j in range(i+1, min(i+20, len(Z))):
            dists.append(np.linalg.norm(Z[i] - Z[j]))
    dists = np.array(dists)
    p10 = np.percentile(dists, 10)
    med_dist = np.median(dists)
    sigma_grp = p10 / 3.0
    sigma_alt = med_dist / np.sqrt(2 * max(eff_rank, 1))
    print("  sigma cal: P10=%.3f -> sigma_grp=%.3f (too tight at d=2)" % (p10, sigma_grp))
    print("  sigma cal: med=%.3f, d_eff=%d -> sigma_alt=%.3f <- USING THIS" % (med_dist, eff_rank, sigma_alt))
    return sigma_alt, eff_rank


def calibrate_action_embedding(encoder, seed):
    global ACTION_EMB
    ACTION_EMB = np.eye(C.k)
    emb_scale = 1.0
    sigma_est, _ = calibrate_sigma(encoder, seed)
    for attempt in range(C.action_sep_max_iter):
        rng = np.random.RandomState(seed + 500)
        xs = rng.randn(200, C.d)
        seps = []
        for x in xs:
            zs = [encoder.encode(x, j) for j in range(C.k)]
            for a in range(C.k):
                for b in range(a+1, C.k):
                    seps.append(np.linalg.norm(zs[a] - zs[b]))
        mean_sep = np.mean(seps)
        ratio = mean_sep / sigma_est if sigma_est > 0 else 0.0
        print("  [Sep iter %d] sep=%.4f, sigma=%.4f, ratio=%.2f" % (attempt+1, mean_sep, sigma_est, ratio))
        if ratio >= C.action_sep_target:
            print("  Action separation OK (ratio=%.2f)" % ratio)
            break
        t = C.action_sep_margin * sigma_est / mean_sep
        emb_scale *= t
        ACTION_EMB = np.eye(C.k) * emb_scale
        sigma_est, _ = calibrate_sigma(encoder, seed)
    return sigma_est, emb_scale


# ============================================================================
#  PART 5 -- EVALUATION
# ============================================================================

def evaluate(agent, encoder, env, ctx):
    X = env.get_test(ctx)
    N = len(X)
    Z_all = np.stack([encoder.encode_batch(X, np.full(N, j, dtype=int))
                      for j in range(C.k)], axis=1)
    Z_flat = Z_all.reshape(N * C.k, -1)
    R = agent.R_eff_batch(Z_flat).reshape(N, C.k)
    return float(np.mean(np.argmax(R, axis=1) == env.correct_actions_batch(X, ctx)))


def classify_center(center, env):
    x_test = center.z[:2].copy()
    act_A = env.correct_action(x_test, 'A')
    act_B = env.correct_action(x_test, 'B')
    if act_A == act_B:
        return 'invariant'
    else:
        return 'context_specific'


# ============================================================================
#  PART 6 -- TRAINING LOOP WITH SNAPSHOTS
# ============================================================================

def run_toy_experiment(seed, tag="full",
                       enable_crystallization=True,
                       enable_crucible=True,
                       enable_agency=True,
                       collect_snapshots=False,
                       quiet=False):
    global ACTION_EMB
    np.random.seed(seed)
    lp = (lambda *a, **kw: None) if quiet else print

    lp("\n" + "="*60)
    lp("  Toy Model -- %s (seed=%d)" % (tag, seed))
    lp("="*60)

    env = ToyEnvironment(seed)
    encoder = ToyEncoder()
    sigma_est, emb_scale = calibrate_action_embedding(encoder, seed)
    base_eval = ToyBaseEvaluator(seed + 2)

    sigma_final, eff_rank = calibrate_sigma(encoder, seed)
    merge_thresh = sigma_final
    lp("  Final: sigma=%.4f, merge=%.4f, eff_rank=%d" % (sigma_final, merge_thresh, eff_rank))

    ibf = IBFAgent(sigma_final, merge_thresh, encoder, base_eval,
                   enable_crystallization, enable_crucible, enable_agency)

    snapshots = {}
    log = {'acc_A': [], 'acc_B': [], 'steps': [],
           'n_centers': [], 'n_cryst': [], 'n_verified': [],
           'k_eff_mean': [], 'entropy_mean': []}

    ctx_map = {'A': 0, 'B': 1}
    total_step = 0
    t0 = time.time()

    if collect_snapshots:
        snapshots['before'] = copy.deepcopy(ibf)

    for phase_name in ['A', 'B']:
        ctx_id = ctx_map[phase_name]
        lp("\n-- Phase %s --" % phase_name)
        ibf.set_context(ctx_id)

        for epoch in range(C.E):
            order = np.random.permutation(C.N_pool)
            ek, eH = [], []

            for idx in order:
                x = env.pool[idx]
                z_all = [encoder.encode(x, j) for j in range(C.k)]
                gt = env.correct_action(x, phase_name)

                ch, Rv, kv, ent = ibf.select_action(z_all, deterministic=False)
                ek.extend(kv.tolist())
                eH.append(ent)
                Ri = 1.0 if ch == gt else 0.0
                Rc = ibf.R_eff(z_all[ch])
                ibf.update(z_all[ch], Ri - Rc, x=x, j_chosen=ch)
                total_step += 1

            ibf.end_epoch()

            # Evaluate -- bypass set_context to preserve crucible_verified flags
            saved_ctx = ibf.current_context
            ibf.current_context = 0
            acc_a = evaluate(ibf, encoder, env, 'A')
            ibf.current_context = 1
            acc_b = evaluate(ibf, encoder, env, 'B')
            ibf.current_context = saved_ctx

            log['steps'].append(total_step)
            log['acc_A'].append(acc_a)
            log['acc_B'].append(acc_b)
            log['n_centers'].append(len(ibf.centers))
            log['n_cryst'].append(ibf.count_crystallized())
            log['n_verified'].append(ibf.count_verified())
            log['k_eff_mean'].append(float(np.mean(ek)) if ek else C.k_0)
            log['entropy_mean'].append(float(np.mean(eH)) if eH else 0.0)

            nc = len(ibf.centers)
            nx = ibf.count_crystallized()
            nv = ibf.count_verified()
            lp("  Ep %2d/%d  %d ctr (%d cryst, %d vrf)  "
               "k=%.2f H=%.3f  Acc_A=%.3f Acc_B=%.3f" %
               (epoch+1, C.E, nc, nx, nv,
                log['k_eff_mean'][-1], log['entropy_mean'][-1], acc_a, acc_b))

            if collect_snapshots:
                if phase_name == 'A' and epoch == 4:
                    snapshots['after_5_epochs'] = copy.deepcopy(ibf)
                if phase_name == 'A' and epoch == C.E - 1:
                    snapshots['end_phase_A'] = copy.deepcopy(ibf)
                if phase_name == 'B' and epoch == 0:
                    snapshots['early_phase_B'] = copy.deepcopy(ibf)
                if phase_name == 'B' and epoch == C.E - 1:
                    snapshots['end_phase_B'] = copy.deepcopy(ibf)

    elapsed = time.time() - t0
    lp("\n  Total time: %.1fs" % elapsed)

    acc_A_end_A = log['acc_A'][C.E - 1]
    acc_A_end_B = log['acc_A'][-1]
    acc_B_end_B = log['acc_B'][-1]
    BT_A = acc_A_end_B - acc_A_end_A

    lp("\n  -- Summary --")
    lp("  Centers: %d (%d cryst, %d verified)" %
       (len(ibf.centers), ibf.count_crystallized(), ibf.count_verified()))
    lp("  Acc_A (end A): %.3f" % acc_A_end_A)
    lp("  Acc_A (end B): %.3f  ->  BT_A = %+.3f" % (acc_A_end_B, BT_A))
    lp("  Acc_B (end B): %.3f" % acc_B_end_B)

    n_inv, n_ctx = 0, 0
    n_cryst_inv, n_cryst_ctx = 0, 0
    n_dissolved_inv, n_dissolved_ctx = 0, 0
    n_verified_inv, n_verified_ctx = 0, 0
    for c in ibf.centers:
        cat = classify_center(c, env)
        is_inv = (cat == 'invariant')
        if is_inv: n_inv += 1
        else:      n_ctx += 1
        if c.is_crystallized():
            if is_inv: n_cryst_inv += 1
            else:      n_cryst_ctx += 1
        if c.is_crystallized() and c.crucible_verified:
            if is_inv: n_verified_inv += 1
            else:      n_verified_ctx += 1
        if c.dissolution_log or (c.was_ever_crystallized and not c.is_crystallized()):
            if is_inv: n_dissolved_inv += 1
            else:      n_dissolved_ctx += 1

    total_dissolved = sum(len(c.dissolution_log) for c in ibf.centers)

    lp("\n  -- Center Classification --")
    lp("  Invariant:     %3d total, %3d cryst, %3d verified, %3d dissolved" %
       (n_inv, n_cryst_inv, n_verified_inv, n_dissolved_inv))
    lp("  Context-spec:  %3d total, %3d cryst, %3d verified, %3d dissolved" %
       (n_ctx, n_cryst_ctx, n_verified_ctx, n_dissolved_ctx))
    lp("  Total dissolution events: %d" % total_dissolved)

    metrics = {
        'acc_A_end_A': acc_A_end_A,
        'acc_A_end_B': acc_A_end_B,
        'acc_B_end_B': acc_B_end_B,
        'BT_A': BT_A,
        'n_centers': len(ibf.centers),
        'n_cryst': ibf.count_crystallized(),
        'n_verified': ibf.count_verified(),
        'n_dissolved': total_dissolved,
        'n_inv': n_inv, 'n_ctx': n_ctx,
        'n_cryst_inv': n_cryst_inv, 'n_cryst_ctx': n_cryst_ctx,
        'n_verified_inv': n_verified_inv, 'n_verified_ctx': n_verified_ctx,
        'n_dissolved_inv': n_dissolved_inv, 'n_dissolved_ctx': n_dissolved_ctx,
    }
    return metrics, log, ibf, env, encoder, base_eval, snapshots


# ============================================================================
#  PART 7 -- RUN FULL MODEL
# ============================================================================

metrics, log, ibf_final, env, encoder, base_eval, snapshots = \
    run_toy_experiment(SEED, tag="full", collect_snapshots=True)

if metrics['n_verified'] == 0:
    print("\n  WARNING: Zero verified centers. Crucible did not fire.")
else:
    print("\n  Crucible active: %d verified, %d dissolution events" %
          (metrics['n_verified'], metrics['n_dissolved']))
    print("    Invariant verified: %d, Context-specific verified: %d" %
          (metrics['n_verified_inv'], metrics['n_verified_ctx']))


# ============================================================================
#  PART 8 -- 8-PANEL FIGURE
#
#  Design:
#   - Global vmax across all landscape panels (panel a reads as flat)
#   - Landscape cmap = PiYG (no red/blue clash with markers)
#   - Marker colors: gold (invariant), purple (ctx-spec), teal (verified),
#     burnt-orange (dissolved) -- ColorBrewer Dark2
#   - Agency cmap = inferno (warm sequential, distinct from landscape)
#   - Decision boundaries in all landscape panels
#   - Panel (e) on white background -- cleanest, most important panel
#   - Marker sizes tuned for two-column print (~1.7in per panel)
# ============================================================================

print("\n" + "="*60)
print("  GENERATING 8-PANEL FIGURE")
print("="*60)

# -- Axis range --
all_pts = np.vstack([env.pool, env.test_A, env.test_B])
pad = 0.3
xlim = (max(all_pts[:, 0].min() - pad, -3.8), min(all_pts[:, 0].max() + pad, 3.8))
ylim = (max(all_pts[:, 1].min() - pad, -3.8), min(all_pts[:, 1].max() + pad, 3.8))

GRID_RES = 120
gx = np.linspace(xlim[0], xlim[1], GRID_RES)
gy = np.linspace(ylim[0], ylim[1], GRID_RES)
GX, GY = np.meshgrid(gx, gy)
grid_pts = np.column_stack([GX.ravel(), GY.ravel()])

# -- Color scheme --
CMAP_LANDSCAPE = 'PiYG'
CMAP_AGENCY    = 'inferno'
CLR_INVARIANT  = '#e6ab02'    # gold
CLR_CTXSPEC    = '#7570b3'    # muted purple
CLR_VERIFIED   = '#1b9e77'    # teal-green
CLR_DISSOLVED  = '#d95f02'    # burnt orange
CLR_SILENCED   = '#cccccc'
CLR_PHASE_B    = '#e7298a'    # magenta-pink
CLR_POOL       = '#bbbbbb'
POOL_ALPHA     = 0.15
POOL_SIZE      = 4

SZ_CRYST     = 55
SZ_TRANSIENT = 12
SZ_VERIFIED  = 80
SZ_DISSOLVED = 65
SZ_PENDING   = 25
SZ_BROADCAST = 80


def compute_Reff_landscape(agent, encoder, ctx_id, grid_pts):
    saved = agent.current_context
    agent.current_context = ctx_id
    N = len(grid_pts)
    Z_all = np.stack([encoder.encode_batch(grid_pts, np.full(N, j, dtype=int))
                      for j in range(C.k)], axis=1)
    Z_flat = Z_all.reshape(N * C.k, -1)
    R = agent.R_eff_batch(Z_flat).reshape(N, C.k)
    margin = R[:, 0] - R[:, 1]
    agent.current_context = saved
    return margin


def compute_keff_map(agent, encoder, grid_pts):
    keffs = np.zeros(len(grid_pts))
    for i, x in enumerate(grid_pts):
        z = encoder.encode(x, 0)
        keffs[i] = agent.k_eff(z)
    return keffs


# -- Pre-compute global vmax from strongest landscape --
agent_endA_tmp = snapshots['end_phase_A']
agent_endA_tmp.current_context = 0
margin_ref = compute_Reff_landscape(agent_endA_tmp, encoder, 0, grid_pts)
GLOBAL_VMAX = max(abs(margin_ref.min()), abs(margin_ref.max()), 0.05)
print("  Global vmax = %.3f (from end-of-Phase-A landscape)" % GLOBAL_VMAX)


def plot_landscape(ax, agent, ctx_id):
    margin = compute_Reff_landscape(agent, encoder, ctx_id, grid_pts)
    cf = ax.contourf(GX, GY, margin.reshape(GX.shape), levels=40,
                     cmap=CMAP_LANDSCAPE, vmin=-GLOBAL_VMAX, vmax=GLOBAL_VMAX,
                     alpha=0.70)
    return cf


def draw_boundaries(ax, alpha=0.30):
    bx = np.linspace(xlim[0], xlim[1], 200)
    ax.plot(bx, bx, '-', color='#555555', alpha=alpha, linewidth=0.6)
    ax.plot(bx, -bx, '-', color='#555555', alpha=alpha, linewidth=0.6)


# -- Build figure --
fig, axes = plt.subplots(2, 4, figsize=(14, 7.5))
plt.subplots_adjust(hspace=0.32, wspace=0.28,
                    left=0.04, right=0.96, bottom=0.08, top=0.89)

for ax in axes.flat:
    ax.set_xlim(xlim)
    ax.set_ylim(ylim)
    ax.set_aspect('equal')
    ax.tick_params(labelsize=6, length=2)
    ax.set_facecolor('#fafafa')


# (a) Empty landscape
ax = axes[0, 0]
plot_landscape(ax, snapshots['before'], 0)
draw_boundaries(ax, alpha=0.15)
ax.scatter(env.pool[:, 0], env.pool[:, 1], s=POOL_SIZE, c=CLR_POOL,
           alpha=POOL_ALPHA, zorder=2, linewidths=0)
ax.set_title("(a) Before interaction", fontsize=9, fontweight='bold')
ax.set_xlabel("x1 (invariant)", fontsize=7)
ax.set_ylabel("x2 (context-specific)", fontsize=7)


# (b) Nucleation (epoch 5)
ax = axes[0, 1]
agent_5 = snapshots['after_5_epochs']
agent_5.current_context = 0
plot_landscape(ax, agent_5, 0)
draw_boundaries(ax, alpha=0.20)
ax.scatter(env.pool[:, 0], env.pool[:, 1], s=POOL_SIZE, c=CLR_POOL,
           alpha=POOL_ALPHA, zorder=2, linewidths=0)
if agent_5.centers:
    pos = np.array([c.z[:2] for c in agent_5.centers])
    vs = np.array([abs(c.v) for c in agent_5.centers])
    sizes = 15 + 80 * vs / (vs.max() + 1e-8)
    ax.scatter(pos[:, 0], pos[:, 1], s=sizes, c=CLR_DISSOLVED,
               edgecolors='black', linewidths=0.4, alpha=0.85, zorder=3)
n5 = len(agent_5.centers)
ax.set_title("(b) Nucleation (epoch 5, %d centers)" % n5, fontsize=9, fontweight='bold')
ax.set_xlabel("x1", fontsize=7)


# (c) End of Phase A -- crystallization
ax = axes[0, 2]
agent_endA = snapshots['end_phase_A']
agent_endA.current_context = 0
plot_landscape(ax, agent_endA, 0)
draw_boundaries(ax)

for c in agent_endA.centers:
    cat = classify_center(c, env)
    clr = CLR_INVARIANT if cat == 'invariant' else CLR_CTXSPEC
    if c.is_crystallized():
        ax.scatter(c.z[0], c.z[1], s=SZ_CRYST, c=clr,
                   edgecolors='black', linewidths=0.6, marker='o', zorder=4)
    else:
        ax.scatter(c.z[0], c.z[1], s=SZ_TRANSIENT, c=clr, alpha=0.45,
                   edgecolors='none', marker='o', zorder=3)

n_cryst_A = agent_endA.count_crystallized()
n_tot_A = len(agent_endA.centers)
ax.set_title("(c) Crystallization (%d/%d cryst)" % (n_cryst_A, n_tot_A),
             fontsize=9, fontweight='bold')
ax.set_xlabel("x1", fontsize=7)


# (d) Context switch -- Phase B epoch 1
ax = axes[0, 3]
agent_earlyB = snapshots['early_phase_B']
agent_earlyB.current_context = 1
plot_landscape(ax, agent_earlyB, 1)
draw_boundaries(ax, alpha=0.20)

for c in agent_earlyB.centers:
    if c.context_id == 0:
        ax.scatter(c.z[0], c.z[1], s=SZ_TRANSIENT + 4, c=CLR_SILENCED,
                   edgecolors='#aaaaaa', linewidths=0.2, alpha=0.45, zorder=3)
for c in agent_earlyB.centers:
    if c.context_id == 1:
        ax.scatter(c.z[0], c.z[1], s=SZ_CRYST - 15, c=CLR_PHASE_B,
                   edgecolors='black', linewidths=0.3, alpha=0.90, zorder=4)

n_a = sum(1 for c in agent_earlyB.centers if c.context_id == 0)
n_b = sum(1 for c in agent_earlyB.centers if c.context_id == 1)
ax.set_title("(d) Context switch (%dA silenced, %dB new)" % (n_a, n_b),
             fontsize=9, fontweight='bold')
ax.set_xlabel("x1", fontsize=7)


# (e) Crucible verdict -- white background, clean
ax = axes[1, 0]
ax.set_facecolor('white')
agent_endB = snapshots['end_phase_B']
draw_boundaries(ax, alpha=0.18)

for c in agent_endB.centers:
    if c.context_id == 1:
        ax.scatter(c.z[0], c.z[1], s=4, c='#dddddd', alpha=0.30,
                   zorder=1, linewidths=0)

n_vrf_plot, n_diss_plot, n_pending_plot = 0, 0, 0
for c in agent_endB.centers:
    if c.context_id != 0:
        continue
    if not c.was_ever_crystallized:
        ax.scatter(c.z[0], c.z[1], s=4, c='#eeeeee', alpha=0.25, zorder=1,
                   linewidths=0)
        continue

    if c.is_crystallized() and c.crucible_verified:
        ax.scatter(c.z[0], c.z[1], s=SZ_VERIFIED, c=CLR_VERIFIED, marker='o',
                   edgecolors='#0a5e3c', linewidths=1.0, zorder=5)
        n_vrf_plot += 1
    elif c.dissolution_log or (c.was_ever_crystallized and not c.is_crystallized()):
        ax.scatter(c.z[0], c.z[1], s=SZ_DISSOLVED, c=CLR_DISSOLVED, marker='X',
                   edgecolors='#8b3a00', linewidths=0.8, zorder=5)
        n_diss_plot += 1
    elif c.is_crystallized() and not c.crucible_verified:
        ax.scatter(c.z[0], c.z[1], s=SZ_PENDING, c='#fee08b', marker='o',
                   edgecolors='#b8860b', linewidths=0.4, zorder=4, alpha=0.7)
        n_pending_plot += 1

ax.set_title("(e) Crucible (%d verified, %d dissolved)" % (n_vrf_plot, n_diss_plot),
             fontsize=9, fontweight='bold')
ax.set_xlabel("x1", fontsize=7)
ax.set_ylabel("x2", fontsize=7)


# (f) Forward transfer -- verified broadcasting into Phase B
ax = axes[1, 1]
agent_endB_f = copy.deepcopy(snapshots['end_phase_B'])
agent_endB_f.current_context = 1
plot_landscape(ax, agent_endB_f, 1)
draw_boundaries(ax, alpha=0.20)

n_broadcasting = 0
for c in agent_endB_f.centers:
    if c.context_id == 0 and c.is_crystallized() and c.crucible_verified:
        ax.scatter(c.z[0], c.z[1], s=SZ_BROADCAST, c=CLR_VERIFIED, marker='o',
                   edgecolors='#0a5e3c', linewidths=1.0, alpha=0.90, zorder=5)
        circle = plt.Circle((c.z[0], c.z[1]), c.sigma * 1.5, fill=False,
                            color=CLR_VERIFIED, alpha=0.30, linewidth=0.9,
                            linestyle='--', zorder=4)
        ax.add_patch(circle)
        n_broadcasting += 1
for c in agent_endB_f.centers:
    if c.context_id == 1 and c.is_crystallized():
        ax.scatter(c.z[0], c.z[1], s=SZ_TRANSIENT, c=CLR_PHASE_B, alpha=0.35,
                   zorder=3, linewidths=0)

ax.set_title("(f) Forward transfer (%d broadcasting)" % n_broadcasting,
             fontsize=9, fontweight='bold')
ax.set_xlabel("x1", fontsize=7)


# (g) Retention -- re-evaluate on context A
ax = axes[1, 2]
agent_endB_g = copy.deepcopy(snapshots['end_phase_B'])
agent_endB_g.current_context = 0
plot_landscape(ax, agent_endB_g, 0)
draw_boundaries(ax)

for c in agent_endB_g.centers:
    if c.context_id == 0 and c.is_crystallized():
        cat = classify_center(c, env)
        clr = CLR_INVARIANT if cat == 'invariant' else CLR_CTXSPEC
        ax.scatter(c.z[0], c.z[1], s=SZ_CRYST, c=clr,
                   edgecolors='black', linewidths=0.5, zorder=4)

acc_A_ret = metrics['acc_A_end_B']
bt = metrics['BT_A']
ax.set_title("(g) Retention: Acc_A=%.2f (BT=%+.2f)" % (acc_A_ret, bt),
             fontsize=9, fontweight='bold')
ax.set_xlabel("x1", fontsize=7)


# (h) Agency heatmap -- inferno, visually distinct
ax = axes[1, 3]
keffs = compute_keff_map(snapshots['end_phase_B'], encoder, grid_pts)
kmin_v, kmax_v = keffs.min(), keffs.max()

if kmax_v - kmin_v < 0.3:
    ax.contourf(GX, GY, keffs.reshape(GX.shape), levels=30,
                cmap=CMAP_AGENCY, alpha=0.75)
    ax.set_title("(h) Agency: k_eff in [%.1f, %.1f] (flat)" % (kmin_v, kmax_v),
                 fontsize=9, fontweight='bold')
    ax.text(0.5, 0.05, "Agency neutral at d=2\nsee chess (S7)",
            transform=ax.transAxes, fontsize=6, ha='center', va='bottom',
            style='italic', color='#444444',
            bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.8))
else:
    im = ax.contourf(GX, GY, keffs.reshape(GX.shape), levels=30,
                     cmap=CMAP_AGENCY, alpha=0.85)
    cb = plt.colorbar(im, ax=ax, shrink=0.65, aspect=15, pad=0.02)
    cb.set_label('k_eff', fontsize=7)
    cb.ax.tick_params(labelsize=6)
    ax.set_title("(h) Agency: k_eff in [%.1f, %.1f]" % (kmin_v, kmax_v),
                 fontsize=9, fontweight='bold')

for c in snapshots['end_phase_B'].centers:
    if c.is_crystallized():
        ax.scatter(c.z[0], c.z[1], s=6, c='white', alpha=0.5, zorder=3,
                   linewidths=0.3, edgecolors='black')

draw_boundaries(ax, alpha=0.15)
ax.set_xlabel("x1", fontsize=7)


# -- Legend --
legend_elements = [
    Line2D([0], [0], marker='o', color='w', markerfacecolor=CLR_INVARIANT,
           markeredgecolor='black', markeredgewidth=0.5,
           markersize=8, label='Invariant crystal', linestyle='None'),
    Line2D([0], [0], marker='o', color='w', markerfacecolor=CLR_CTXSPEC,
           markeredgecolor='black', markeredgewidth=0.5,
           markersize=8, label='Context-specific crystal', linestyle='None'),
    Line2D([0], [0], marker='o', color='w', markerfacecolor=CLR_VERIFIED,
           markeredgecolor='#0a5e3c', markeredgewidth=0.8,
           markersize=8, label='Verified (survived crucible)', linestyle='None'),
    Line2D([0], [0], marker='X', color='w', markerfacecolor=CLR_DISSOLVED,
           markeredgecolor='#8b3a00', markeredgewidth=0.6,
           markersize=8, label='Dissolved (contradicted)', linestyle='None'),
    Line2D([0], [0], marker='o', color='w', markerfacecolor=CLR_SILENCED,
           markeredgecolor='#aaaaaa', markeredgewidth=0.3,
           markersize=7, label='Silenced (cross-ctx)', linestyle='None'),
]
fig.legend(handles=legend_elements, loc='lower center', ncol=5,
           fontsize=7, frameon=True, framealpha=0.9,
           edgecolor='#cccccc', bbox_to_anchor=(0.5, -0.005))

fig.suptitle("Figure 1 -- The IBF Lifecycle in Two Dimensions",
             fontsize=12, fontweight='bold', y=0.94)

fig_path_png = os.path.join(OUTPUT_DIR, "fig1_toy_model.png")
fig_path_pdf = os.path.join(OUTPUT_DIR, "fig1_toy_model.pdf")
fig.savefig(fig_path_png, dpi=300, bbox_inches='tight')
fig.savefig(fig_path_pdf, bbox_inches='tight')
plt.close(fig)
print("  Saved: %s" % fig_path_png)
print("  Saved: %s" % fig_path_pdf)


# ============================================================================
#  PART 9 -- MINI ABLATION TABLE
# ============================================================================

print("\n" + "="*60)
print("  MINI ABLATION TABLE")
print("="*60)

ablation_results = {}
ablation_results['Full IBF'] = metrics

m_na, *_ = run_toy_experiment(SEED, tag="No-Agency",
                               enable_agency=False, quiet=True)
ablation_results['No-Agency'] = m_na

m_nc, *_ = run_toy_experiment(SEED, tag="No-Cryst",
                               enable_crystallization=False, quiet=True)
ablation_results['No-Cryst'] = m_nc

m_ncr, *_ = run_toy_experiment(SEED, tag="No-Crucible",
                                enable_crucible=False, quiet=True)
ablation_results['No-Crucible'] = m_ncr

np.random.seed(SEED)
env_p = ToyEnvironment(SEED)
enc_p = ToyEncoder()
calibrate_action_embedding(enc_p, SEED)
be_p = ToyBaseEvaluator(SEED + 2)
passive = PassiveAgent(enc_p, be_p)
acc_A_passive = evaluate(passive, enc_p, env_p, 'A')
acc_B_passive = evaluate(passive, enc_p, env_p, 'B')
ablation_results['Passive'] = {
    'acc_A_end_A': acc_A_passive,
    'acc_A_end_B': acc_A_passive,
    'acc_B_end_B': acc_B_passive,
    'BT_A': 0.0,
    'n_verified': 0, 'n_dissolved': 0,
    'n_verified_inv': 0, 'n_verified_ctx': 0,
    'n_dissolved_inv': 0, 'n_dissolved_ctx': 0,
}

print("\n  %-15s %-14s %-15s %-10s %-10s" %
      ('Condition', 'Acc_A(end A)', 'Acc_A(after B)', 'BT_A', 'Acc_B'))
print("  " + "-"*64)
for name, m in ablation_results.items():
    print("  %-15s %-14.3f %-15.3f %+-10.3f %-10.3f" %
          (name, m['acc_A_end_A'], m['acc_A_end_B'], m['BT_A'],
           m.get('acc_B_end_B', 0)))


# ============================================================================
#  PART 10 -- DETAILED TELEMETRY + CURVES
# ============================================================================

print("\n" + "="*60)
print("  DETAILED TELEMETRY (Full IBF)")
print("="*60)

m = metrics
print("  sigma_base: %.4f" % ibf_final.sigma_base)
print("  Centers nucleated: %d" % m['n_centers'])
print("  Centers crystallized: %d" % m['n_cryst'])
print("    Invariant: %d" % m['n_cryst_inv'])
print("    Context-specific: %d" % m['n_cryst_ctx'])
print("  Crucible verified: %d" % m['n_verified'])
print("    Invariant verified: %d" % m['n_verified_inv'])
print("    Context-specific verified: %d" % m['n_verified_ctx'])
print("  Dissolution events: %d" % m['n_dissolved'])
print("    Invariant dissolved: %d" % m.get('n_dissolved_inv', 0))
print("    Context-specific dissolved: %d" % m.get('n_dissolved_ctx', 0))
print("  Acc_A (end Phase A): %.3f" % m['acc_A_end_A'])
print("  Acc_A (end Phase B): %.3f" % m['acc_A_end_B'])
print("  BT_A: %+.3f" % m['BT_A'])
print("  Acc_B (end Phase B): %.3f" % m['acc_B_end_B'])

keffs_all = []
for x in env.pool:
    z = encoder.encode(x, 0)
    keffs_all.append(snapshots['end_phase_B'].k_eff(z))
keffs_all = np.array(keffs_all)
print("  k_eff range: [%.2f, %.2f]" % (keffs_all.min(), keffs_all.max()))
print("  k_eff mean: %.2f, std: %.2f" % (keffs_all.mean(), keffs_all.std()))

all_diss = []
for ci, c in enumerate(ibf_final.centers):
    for evt in c.dissolution_log:
        cat = classify_center(c, env)
        evt_copy = dict(evt)
        evt_copy['center_idx'] = ci
        evt_copy['category'] = cat
        all_diss.append(evt_copy)
if all_diss:
    print("\n  -- Dissolution Events (%d) --" % len(all_diss))
    for ev in sorted(all_diss, key=lambda e: e['step'])[:20]:
        print("    step=%5d cat=%-18s v=%+.4f mu_D=%+.4f prod=%+.4f n_cross=%d" %
              (ev['step'], ev['category'], ev['v'], ev['mu_D_recent'],
               ev['product'], ev['n_cross']))
    if len(all_diss) > 20:
        print("    ... (%d more)" % (len(all_diss) - 20))

# -- Learning curves --
fig2, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
epochs = np.arange(1, len(log['acc_A']) + 1)
ax1.plot(epochs, log['acc_A'], 'b-', label='Acc_A', linewidth=1.5)
ax1.plot(epochs, log['acc_B'], 'r-', label='Acc_B', linewidth=1.5)
ax1.axvline(x=C.E, color='gray', linestyle='--', alpha=0.5, label='Phase boundary')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Accuracy')
ax1.set_title('Learning Curves')
ax1.legend(fontsize=8)
ax1.grid(True, alpha=0.3)

ax2.plot(epochs, log['n_cryst'], 'g-', label='Crystallized', linewidth=1.5)
ax2.plot(epochs, log['n_verified'], 'm-', label='Verified', linewidth=1.5)
ax2.plot(epochs, log['n_centers'], 'k--', label='Total centers', linewidth=1.0, alpha=0.5)
ax2.axvline(x=C.E, color='gray', linestyle='--', alpha=0.5)
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Count')
ax2.set_title('Center Lifecycle')
ax2.legend(fontsize=8)
ax2.grid(True, alpha=0.3)

fig2.tight_layout()
curves_path = os.path.join(OUTPUT_DIR, "toy_model_curves.png")
fig2.savefig(curves_path, dpi=150, bbox_inches='tight')
plt.close(fig2)
print("\n  Saved: %s" % curves_path)

# -- Agency + entropy --
fig3, (ax3, ax4) = plt.subplots(1, 2, figsize=(10, 4))
ax3.plot(epochs, log['k_eff_mean'], 'g-', linewidth=1.5)
ax3.axvline(x=C.E, color='gray', linestyle='--', alpha=0.5)
ax3.set_xlabel('Epoch')
ax3.set_ylabel('Mean k_eff')
ax3.set_title('Agency Modulation')
ax3.grid(True, alpha=0.3)

ax4.plot(epochs, log['entropy_mean'], 'purple', linewidth=1.5)
ax4.axvline(x=C.E, color='gray', linestyle='--', alpha=0.5)
ax4.set_xlabel('Epoch')
ax4.set_ylabel('Mean Entropy')
ax4.set_title('Exploration (Shannon Entropy)')
ax4.grid(True, alpha=0.3)

fig3.tight_layout()
agency_path = os.path.join(OUTPUT_DIR, "toy_model_agency.png")
fig3.savefig(agency_path, dpi=150, bbox_inches='tight')
plt.close(fig3)
print("  Saved: %s" % agency_path)

print("\n" + "="*60)
print("  TOY MODEL COMPLETE")
print("="*60)

IBF 2D Toy Model -- Figure 1 Generator
  d=2, k=2, N_pool=200, E=25, N_test=500
  eta=0.10, mu_base=0.06, mu_cryst=0.001
  n_cross_min=4, rev_thresh=-0.125

  Toy Model -- full (seed=42)
  sigma cal: P10=0.602 -> sigma_grp=0.201 (too tight at d=2)
  sigma cal: med=1.628, d_eff=2 -> sigma_alt=0.814 <- USING THIS
  [Sep iter 1] sep=1.4142, sigma=0.8141, ratio=1.74
  sigma cal: P10=0.602 -> sigma_grp=0.201 (too tight at d=2)
  sigma cal: med=1.628, d_eff=2 -> sigma_alt=0.814 <- USING THIS
  [Sep iter 2] sep=3.2564, sigma=0.8141, ratio=4.00
  Action separation OK (ratio=4.00)
  Base evaluator spread: 0.0527 (variance=0.05)
  sigma cal: P10=0.602 -> sigma_grp=0.201 (too tight at d=2)
  sigma cal: med=1.628, d_eff=2 -> sigma_alt=0.814 <- USING THIS
  Final: sigma=0.8141, merge=0.8141, eff_rank=2

-- Phase A --
  Ep  1/25  18 ctr (4 cryst, 0 vrf)  k=5.00 H=0.522  Acc_A=0.932 Acc_B=0.488
  Ep  2/25  21 ctr (12 cryst, 0 vrf)  k=5.16 H=0.335  Acc_A=0.950 Acc_B=0.488
  Ep  3/25  22 ctr (15 cryst,

## Reading the Output: The Seven Steps

The printout above walks through the IBF lifecycle. Here is what to look for at each stage, mapped to the paper's narrative.

### Step 1 — The Empty Landscape (§2.2)
Before any interaction, the coherence landscape is flat: `R_eff ≈ 0.5` everywhere. The system selects actions at random. **Look for:** `Acc_A ≈ 0.50` at the start (the Passive baseline).

### Step 2 — Nucleation (§2.3)
Where the discrepancy signal `D = R_imposed − R_chosen` is large and no existing center covers the region, a new center nucleates (Eq. 2). **Look for:** centers appearing in the first few epochs (e.g., `22 ctr` by epoch 5).

### Step 3 — Crystallization (§2.4)
Centers that receive repeated, consistent D-signals undergo a stability transition: their decay rate drops by 60× (`μ_eff: 0.06 → 0.001`). **Look for:** `18 cryst` by the end of Phase A, and `Acc_A ≈ 0.95`.

### Step 4 — Context Switch and Isolation (§2.5)
Phase B begins. Context-gated read access silences all Phase A centers (`γ_i = 0`). They remain in the landscape but contribute nothing. **Look for:** new centers nucleating in Phase B epoch 1, while the Phase A count stays frozen.

### Step 5 — The Crucible (§2.6)
Phase B inputs activate the kernel near sleeping Phase A crystals. Each is tested: does its stored correction agree with the new context's D-signal?
- **Invariant crystals** (x₁-encoding): consistent → survive → earn broadcast rights (`crucible_verified = True`)
- **Context-specific crystals** (x₂-encoding): contradicted → dissolve (`μ_eff` reverts to 0.06)

**Look for:** `14 verified` and dissolution events in the telemetry. The paper reports 14 verified and 4 unique centers dissolved.

### Step 6 — Forward Transfer and Retention (§2.7)
Verified universals broadcast into Phase B, contributing x₁-structure learned in Phase A. **Look for:** `Acc_B = 0.970`. Return to Phase A evaluation: `Acc_A = 0.778`, giving `BT_A = −0.174`. This is the thermodynamic cost of enabling cross-context transfer.

### Step 7 — Emergent Agency (§2.8)
Each crystallized center accumulates an agency weight based on D-signal variance. Low variance → high `k_eff` (commit). High variance → low `k_eff` (hedge). **Look for:** `k_eff range: [5.00, 9.84]` in the telemetry — spatially nonuniform responsiveness that emerged from interaction.

### Mini-Ablation (§2.9)
The ablation table isolates each mechanism's contribution:
- **No-Crucible**: `BT_A = 0.000` — perfect isolation, zero transfer. Confirms safety is structural.
- **No-Agency**: Slightly better retention than full IBF in this adversarial regime (predicted).
- **Passive**: Chance performance throughout. No learning without modification dynamics.

In [ ]:
## Figure Generation

The cell below produces the 8-panel figure for the paper:
- **8 individual PDFs** (`fig1_a.pdf` through `fig1_h.pdf`) for LaTeX `\subfigure` inclusion
- **1 composite PNG** (`fig1_composite.png`) for presentations

A LaTeX snippet with auto-filled caption is printed at the end.

In [23]:
# ============================================================================
#  FIGURE 1 v5 -- 8 separate panel PDFs + 1 composite PNG
#  Run AFTER the main toy model cell (needs: snapshots, env, encoder, metrics)
#
#  NOTE: All strings avoid $ and {} to prevent artifact storage corruption.
#  LaTeX math is built via variables, never inline in function calls.
# ============================================================================

import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
import copy, os

plt.rcParams['font.family'] = 'serif'
plt.rcParams['font.serif'] = ['Computer Modern Roman', 'Times New Roman', 'DejaVu Serif']
plt.rcParams['mathtext.fontset'] = 'cm'
plt.rcParams['axes.unicode_minus'] = False

OUTPUT_DIR = "toy_model_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Pre-build all LaTeX label strings as variables (avoids brace corruption)
LAB_X1 = r'$x_1$'
LAB_X2 = r'$x_2$'
LAB_X1_FULL = r'$x_1$ (invariant)'
LAB_X2_FULL = r'$x_2$ (context-specific)'
LAB_KEFF = r'$k_{\mathrm{eff}}$'
LAB_ACCA = r'Acc$_A$'

print("="*60)
print("  GENERATING 8 SEPARATE PANEL PDFs")
print("="*60)

# -- Geometry --
all_pts = np.vstack([env.pool, env.test_A, env.test_B])
pad = 0.3
xlim = (max(all_pts[:, 0].min() - pad, -3.8), min(all_pts[:, 0].max() + pad, 3.8))
ylim = (max(all_pts[:, 1].min() - pad, -3.8), min(all_pts[:, 1].max() + pad, 3.8))

GRID_RES = 150
gx = np.linspace(xlim[0], xlim[1], GRID_RES)
gy = np.linspace(ylim[0], ylim[1], GRID_RES)
GX, GY = np.meshgrid(gx, gy)
grid_pts = np.column_stack([GX.ravel(), GY.ravel()])

# -- Colors --
CLR_INVARIANT  = '#e6ab02'
CLR_CTXSPEC    = '#7570b3'
CLR_VERIFIED   = '#1b9e77'
CLR_DISSOLVED  = '#d95f02'
CLR_SILENCED   = '#cccccc'
CLR_PHASE_B    = '#e7298a'

SZ_CRYST     = 55
SZ_TRANS     = 8
SZ_VERIFIED  = 80
SZ_DISSOLVED = 60
SZ_BROADCAST = 75

CMAP_ACC = LinearSegmentedColormap.from_list('acc',
    ['#f4a0a0', '#fce4e4', '#f5f5f5', '#e0f0e0', '#7dc97d'], N=256)
CMAP_KEFF = LinearSegmentedColormap.from_list('keff',
    ['#3b4cc0', '#6b8fd4', '#c0c0c0', '#e8845a', '#b40426'], N=256)

PANEL_SIZE = (3.5, 3.5)


def compute_accuracy_margin(agent, ctx_id, ctx_name):
    saved = agent.current_context
    agent.current_context = ctx_id
    N = len(grid_pts)
    Z_all = np.stack([encoder.encode_batch(grid_pts, np.full(N, j, dtype=int))
                      for j in range(C.k)], axis=1)
    Z_flat = Z_all.reshape(N * C.k, -1)
    R = agent.R_eff_batch(Z_flat).reshape(N, C.k)
    correct = env.correct_actions_batch(grid_pts, ctx_name)
    margin = np.zeros(N)
    for i in range(N):
        margin[i] = R[i, correct[i]] - R[i, 1 - correct[i]]
    agent.current_context = saved
    return margin


def plot_acc_bg(ax, agent, ctx_id, ctx_name):
    margin = compute_accuracy_margin(agent, ctx_id, ctx_name)
    vmax = max(abs(margin.min()), abs(margin.max()), 0.05)
    ax.contourf(GX, GY, margin.reshape(GX.shape), levels=40,
                cmap=CMAP_ACC, vmin=-vmax, vmax=vmax, alpha=0.70)


def compute_keff_map(agent):
    k = np.zeros(len(grid_pts))
    for i, x in enumerate(grid_pts):
        z = encoder.encode(x, 0)
        k[i] = agent.k_eff(z)
    return k


def make_panel(full_x=False, full_y=False):
    fig, ax = plt.subplots(1, 1, figsize=PANEL_SIZE)
    ax.set_xlim(xlim)
    ax.set_ylim(ylim)
    ax.set_aspect('equal')
    ax.tick_params(labelsize=7, length=2, width=0.5)
    ax.set_facecolor('#fafafa')
    if full_x:
        ax.set_xlabel(LAB_X1_FULL, fontsize=8)
    else:
        ax.set_xlabel(LAB_X1, fontsize=8)
    if full_y:
        ax.set_ylabel(LAB_X2_FULL, fontsize=8)
    else:
        ax.set_ylabel(LAB_X2, fontsize=8)
    return fig, ax


def save_panel(fig, name):
    path_pdf = os.path.join(OUTPUT_DIR, name + '.pdf')
    path_png = os.path.join(OUTPUT_DIR, name + '.png')
    fig.savefig(path_pdf, bbox_inches='tight')
    fig.savefig(path_png, dpi=300, bbox_inches='tight')
    plt.close(fig)
    print("  %s" % name)
    return path_pdf


def pick_center(centers, cat, quadrant=None):
    cands = [c for c in centers if c.is_crystallized()
             and classify_center(c, env) == cat
             and abs(c.z[0]) < 3.0 and abs(c.z[1]) < 3.0]
    if not cands:
        cands = [c for c in centers if c.is_crystallized()
                 and classify_center(c, env) == cat]
    if not cands:
        return None
    if quadrant:
        qc = []
        for c in cands:
            x, y = c.z[0], c.z[1]
            if quadrant == 1 and x > 0 and y > 0: qc.append(c)
            if quadrant == 2 and x < 0 and y > 0: qc.append(c)
            if quadrant == 3 and x < 0 and y < 0: qc.append(c)
            if quadrant == 4 and x > 0 and y < 0: qc.append(c)
        if qc:
            cands = qc
    cands.sort(key=lambda c: c.z[0]**2 + c.z[1]**2)
    return cands[len(cands) // 3]


# =====================================================================
#  (a) Empty landscape
# =====================================================================
fig, ax = make_panel(full_x=True, full_y=True)
ax.set_facecolor('#f0f0f0')
ax.scatter(env.pool[:, 0], env.pool[:, 1], s=5, c='#999999',
           alpha=0.30, zorder=2, linewidths=0)
ax.text(0.50, 0.50, "~50%% accuracy\n(chance)",
        transform=ax.transAxes, fontsize=10, ha='center', va='center',
        color='#777777',
        bbox=dict(boxstyle='round,pad=0.3', fc='white', ec='#cccccc', alpha=0.9),
        zorder=5)
save_panel(fig, 'fig1_a')


# =====================================================================
#  (b) Nucleation (epoch 5)
#  ALL centers transient: hollow circles, sized by |v|
# =====================================================================
fig, ax = make_panel()
agent_5 = snapshots['after_5_epochs']
agent_5.current_context = 0
plot_acc_bg(ax, agent_5, 0, 'A')

if agent_5.centers:
    vmax_b = max(abs(c.v) for c in agent_5.centers) + 1e-8
    for c in agent_5.centers:
        cat = classify_center(c, env)
        clr = CLR_INVARIANT if cat == 'invariant' else CLR_CTXSPEC
        sz = 20 + 55 * abs(c.v) / vmax_b
        ax.scatter(c.z[0], c.z[1], s=sz,
                   facecolors='none', edgecolors=clr,
                   linewidths=1.0, alpha=0.85, zorder=3)
save_panel(fig, 'fig1_b')


# =====================================================================
#  (c) Crystallization (end Phase A)
#  Crystallized = filled circles, thick edge. Transient = hollow.
# =====================================================================
fig, ax = make_panel()
agent_endA = snapshots['end_phase_A']
agent_endA.current_context = 0
plot_acc_bg(ax, agent_endA, 0, 'A')

for c in agent_endA.centers:
    cat = classify_center(c, env)
    clr = CLR_INVARIANT if cat == 'invariant' else CLR_CTXSPEC
    if c.is_crystallized():
        ax.scatter(c.z[0], c.z[1], s=SZ_CRYST, c=clr,
                   edgecolors='black', linewidths=1.2, zorder=4)
    else:
        ax.scatter(c.z[0], c.z[1], s=SZ_CRYST - 15,
                   facecolors='none', edgecolors=clr,
                   linewidths=0.8, alpha=0.6, zorder=3)

ic = pick_center(agent_endA.centers, 'invariant', quadrant=1)
cc_ann = pick_center(agent_endA.centers, 'context_specific', quadrant=3)
if ic:
    ax.annotate("invariant", xy=(ic.z[0], ic.z[1]),
                xytext=(min(ic.z[0] + 1.2, 3.0), min(ic.z[1] + 1.0, 3.2)),
                fontsize=7, color='#8a6d00',
                arrowprops=dict(arrowstyle='-', color='#8a6d00', lw=0.6),
                zorder=10)
if cc_ann:
    ax.annotate("context-specific", xy=(cc_ann.z[0], cc_ann.z[1]),
                xytext=(max(cc_ann.z[0] + 0.8, -1.5), max(cc_ann.z[1] - 1.0, -3.2)),
                fontsize=7, color='#4a46a3',
                arrowprops=dict(arrowstyle='-', color='#4a46a3', lw=0.6),
                zorder=10)
save_panel(fig, 'fig1_c')


# =====================================================================
#  (d) Context switch
#  Phase A: gray, filled if crystallized, hollow if transient.
#  Phase B: hollow pink (just nucleated = transient).
# =====================================================================
fig, ax = make_panel()
agent_earlyB = snapshots['early_phase_B']
agent_earlyB.current_context = 1
plot_acc_bg(ax, agent_earlyB, 1, 'B')

for c in agent_earlyB.centers:
    if c.context_id == 0:
        if c.is_crystallized():
            ax.scatter(c.z[0], c.z[1], s=SZ_CRYST - 15, c=CLR_SILENCED,
                       edgecolors='#999999', linewidths=0.8, alpha=0.50, zorder=3)
        else:
            ax.scatter(c.z[0], c.z[1], s=SZ_TRANS + 3,
                       facecolors='none', edgecolors='#bbbbbb',
                       linewidths=0.5, alpha=0.35, zorder=3)

for c in agent_earlyB.centers:
    if c.context_id == 1:
        ax.scatter(c.z[0], c.z[1], s=SZ_CRYST - 15,
                   facecolors='none', edgecolors=CLR_PHASE_B,
                   linewidths=1.0, alpha=0.90, zorder=4)
save_panel(fig, 'fig1_d')


# =====================================================================
#  (e) Crucible
#  Phase B accuracy map background (same as d, f).
#  Survived = filled teal + kernel zone on green background.
#  Dissolved = hollow orange + small x + kernel zone on pink background.
#  The landscape explains the verdict spatially.
# =====================================================================
fig, ax = make_panel()
agent_endB = snapshots['end_phase_B']
agent_e = copy.deepcopy(agent_endB)
agent_e.current_context = 1
plot_acc_bg(ax, agent_e, 1, 'B')

# Phase B centers faintly for context
for c in agent_endB.centers:
    if c.context_id == 1 and c.is_crystallized():
        ax.scatter(c.z[0], c.z[1], s=SZ_TRANS, c=CLR_PHASE_B,
                   alpha=0.20, zorder=2, linewidths=0)

n_vrf, n_diss = 0, 0
for c in agent_endB.centers:
    if c.context_id != 0 or not c.was_ever_crystallized:
        continue

    is_verified = c.is_crystallized() and c.crucible_verified
    is_dissolved = c.dissolution_log or (c.was_ever_crystallized and not c.is_crystallized())

    if is_verified:
        kernel_circle = plt.Circle(
            (c.z[0], c.z[1]), c.sigma * 1.8,
            fill=True, facecolor=CLR_VERIFIED, alpha=0.08,
            edgecolor=CLR_VERIFIED, linewidth=0.6, linestyle='-',
            zorder=3)
        ax.add_patch(kernel_circle)
        ax.scatter(c.z[0], c.z[1], s=SZ_VERIFIED, c=CLR_VERIFIED, marker='o',
                   edgecolors='#0a5e3c', linewidths=1.0, zorder=5)
        n_vrf += 1

    elif is_dissolved:
        kernel_circle = plt.Circle(
            (c.z[0], c.z[1]), c.sigma * 1.8,
            fill=True, facecolor=CLR_DISSOLVED, alpha=0.06,
            edgecolor=CLR_DISSOLVED, linewidth=0.5, linestyle='--',
            zorder=3)
        ax.add_patch(kernel_circle)
        ax.scatter(c.z[0], c.z[1], s=SZ_DISSOLVED,
                   facecolors='none', edgecolors=CLR_DISSOLVED,
                   linewidths=1.5, marker='o', zorder=5)
        ax.scatter(c.z[0], c.z[1], s=18, c=CLR_DISSOLVED,
                   marker='x', linewidths=1.0, zorder=6)
        n_diss += 1

save_panel(fig, 'fig1_e')
print("  Panel (e): %d verified, %d dissolved" % (n_vrf, n_diss))


# =====================================================================
#  (f) Forward transfer
# =====================================================================
fig, ax = make_panel()
agent_f = copy.deepcopy(snapshots['end_phase_B'])
agent_f.current_context = 1
plot_acc_bg(ax, agent_f, 1, 'B')

n_bc = 0
for c in agent_f.centers:
    if c.context_id == 0 and c.is_crystallized() and c.crucible_verified:
        ax.scatter(c.z[0], c.z[1], s=SZ_BROADCAST, c=CLR_VERIFIED, marker='o',
                   edgecolors='#0a5e3c', linewidths=0.8, alpha=0.90, zorder=5)
        circle = plt.Circle((c.z[0], c.z[1]), c.sigma * 1.5, fill=False,
                            color=CLR_VERIFIED, alpha=0.25, linewidth=0.7,
                            linestyle='--', zorder=4)
        ax.add_patch(circle)
        n_bc += 1

for c in agent_f.centers:
    if c.context_id == 1 and c.is_crystallized():
        ax.scatter(c.z[0], c.z[1], s=SZ_TRANS, c=CLR_PHASE_B, alpha=0.35,
                   zorder=3, linewidths=0)
save_panel(fig, 'fig1_f')


# =====================================================================
#  (g) Retention -- same accuracy map style as (c), no annotation.
#  The visual comparison between (c) and (g) IS the retention story.
#  Caption provides the numbers.
# =====================================================================
fig, ax = make_panel()
agent_g = copy.deepcopy(snapshots['end_phase_B'])
agent_g.current_context = 0
plot_acc_bg(ax, agent_g, 0, 'A')

for c in agent_g.centers:
    if c.context_id == 0 and c.is_crystallized():
        cat = classify_center(c, env)
        clr = CLR_INVARIANT if cat == 'invariant' else CLR_CTXSPEC
        ax.scatter(c.z[0], c.z[1], s=SZ_CRYST, c=clr,
                   edgecolors='black', linewidths=0.8, zorder=4)

save_panel(fig, 'fig1_g')


# =====================================================================
#  (h) Agency
# =====================================================================
fig, ax = make_panel(full_x=True, full_y=True)
keffs = compute_keff_map(snapshots['end_phase_B'])
kmin_v, kmax_v = keffs.min(), keffs.max()

if kmax_v - kmin_v < 0.3:
    ax.contourf(GX, GY, keffs.reshape(GX.shape), levels=30,
                cmap=CMAP_KEFF, alpha=0.75)
else:
    im = ax.contourf(GX, GY, keffs.reshape(GX.shape), levels=30,
                     cmap=CMAP_KEFF, alpha=0.85)
    cb = plt.colorbar(im, ax=ax, shrink=0.75, aspect=20, pad=0.02)
    cb.set_label(LAB_KEFF, fontsize=9)
    cb.ax.tick_params(labelsize=6)

for c in snapshots['end_phase_B'].centers:
    if c.is_crystallized():
        ax.scatter(c.z[0], c.z[1], s=5, c='white', alpha=0.5, zorder=3,
                   linewidths=0.3, edgecolors='black')
save_panel(fig, 'fig1_h')


# =====================================================================
#  Composite PNG (for presentations)
# =====================================================================
print("\n  Assembling composite...")

from matplotlib.image import imread

panel_names = ['fig1_a', 'fig1_b', 'fig1_c', 'fig1_d',
               'fig1_e', 'fig1_f', 'fig1_g', 'fig1_h']
imgs = []
for name in panel_names:
    path = os.path.join(OUTPUT_DIR, name + '.png')
    imgs.append(imread(path))

fig_comp, axes_comp = plt.subplots(2, 4, figsize=(16, 8))
plt.subplots_adjust(hspace=0.05, wspace=0.05,
                    left=0.01, right=0.99, bottom=0.01, top=0.95)

labels = ['a', 'b', 'c', 'd', 'e', 'f', 'g', 'h']
for idx, ax in enumerate(axes_comp.flat):
    ax.imshow(imgs[idx])
    ax.axis('off')
    ax.text(0.03, 0.97, '(%s)' % labels[idx],
            transform=ax.transAxes, fontsize=12, fontweight='bold',
            va='top', ha='left', color='black',
            bbox=dict(boxstyle='round,pad=0.15', fc='white', ec='none', alpha=0.8))

fig_comp.suptitle("Figure 1 -- The IBF Lifecycle in Two Dimensions",
                  fontsize=14, fontweight='bold', y=0.98)

comp_path = os.path.join(OUTPUT_DIR, "fig1_composite.png")
fig_comp.savefig(comp_path, dpi=200, bbox_inches='tight')
plt.close(fig_comp)
print("  %s" % comp_path)

plt.rcParams['font.family'] = 'sans-serif'

# -- LaTeX snippet --
print("\n" + "="*60)
print("  LaTeX inclusion:")
print("="*60)
print(r"""
\begin{figure*}[t]
\centering
\subfigure[]{\includegraphics[width=0.235\textwidth]{figures/fig1_a.pdf}\label{fig:toy:a}}%
\hfill
\subfigure[]{\includegraphics[width=0.235\textwidth]{figures/fig1_b.pdf}\label{fig:toy:b}}%
\hfill
\subfigure[]{\includegraphics[width=0.235\textwidth]{figures/fig1_c.pdf}\label{fig:toy:c}}%
\hfill
\subfigure[]{\includegraphics[width=0.235\textwidth]{figures/fig1_d.pdf}\label{fig:toy:d}}%
\\[4pt]
\subfigure[]{\includegraphics[width=0.235\textwidth]{figures/fig1_e.pdf}\label{fig:toy:e}}%
\hfill
\subfigure[]{\includegraphics[width=0.235\textwidth]{figures/fig1_f.pdf}\label{fig:toy:f}}%
\hfill
\subfigure[]{\includegraphics[width=0.235\textwidth]{figures/fig1_g.pdf}\label{fig:toy:g}}%
\hfill
\subfigure[]{\includegraphics[width=0.235\textwidth]{figures/fig1_h.pdf}\label{fig:toy:h}}%
""")

print("\n  Caption (auto-filled):")
print("  " + "-"*50)
cap = (
    "\\caption{\\textbf{The IBF lifecycle in two dimensions.} "
    "A system with two input features ($x_1$ invariant, $x_2$ context-specific) "
    "and two actions learns across two sequential contexts where the $x_2$ component "
    "flips sign. Green background = correct classification; pink = incorrect. "
    "(a)~Before interaction: flat landscape, chance performance. "
    "(b)~After 5 epochs: %d memory centers nucleate (hollow circles = transient). "
    "(c)~End of Phase~A: %d of %d centers crystallize (filled circles). "
    "Orange: invariant. Purple: context-specific. "
    "(d)~Context switch: Phase~A crystals silenced (gray), Phase~B centers nucleate (hollow pink). "
    "(e)~The Crucible: Phase~A crystals tested against Phase~B landscape. "
    "%d verified (filled teal with kernel zones in green regions), "
    "%d dissolved (hollow orange in pink regions). "
    "(f)~Verified universals broadcast into Phase~B (dashed circles: kernel influence). "
    "(g)~Return to Phase~A: crystals intact, "
    "$\\mathrm{Acc}_A\\!=\\!%.3f$ ($\\mathrm{BT}_A\\!=\\!%.3f$). "
    "(h)~Emergent agency: $k_{\\mathrm{eff}}$ ranges from $%.1f$ to $%.1f$.}"
)
print(cap % (
    len(snapshots['after_5_epochs'].centers),
    snapshots['end_phase_A'].count_crystallized(),
    len(snapshots['end_phase_A'].centers),
    metrics['n_verified'],
    metrics['n_dissolved'],
    metrics['acc_A_end_B'],
    metrics['BT_A'],
    kmin_v, kmax_v
))
print("\\label{fig:toy}")
print("\\end{figure*}")

print("\n" + "="*60)
print("  DONE -- 8 PDFs + composite PNG")
print("="*60)

  GENERATING 8 SEPARATE PANEL PDFs
  fig1_a
  fig1_b
  fig1_c
  fig1_d
  fig1_e
  Panel (e): 14 verified, 4 dissolved
  fig1_f
  fig1_g
  fig1_h

  Assembling composite...
  toy_model_outputs/fig1_composite.png

  LaTeX inclusion:

\begin{figure*}[t]
\centering
\subfigure[]{\includegraphics[width=0.235\textwidth]{figures/fig1_a.pdf}\label{fig:toy:a}}%
\hfill
\subfigure[]{\includegraphics[width=0.235\textwidth]{figures/fig1_b.pdf}\label{fig:toy:b}}%
\hfill
\subfigure[]{\includegraphics[width=0.235\textwidth]{figures/fig1_c.pdf}\label{fig:toy:c}}%
\hfill
\subfigure[]{\includegraphics[width=0.235\textwidth]{figures/fig1_d.pdf}\label{fig:toy:d}}%
\\[4pt]
\subfigure[]{\includegraphics[width=0.235\textwidth]{figures/fig1_e.pdf}\label{fig:toy:e}}%
\hfill
\subfigure[]{\includegraphics[width=0.235\textwidth]{figures/fig1_f.pdf}\label{fig:toy:f}}%
\hfill
\subfigure[]{\includegraphics[width=0.235\textwidth]{figures/fig1_g.pdf}\label{fig:toy:g}}%
\hfill
\subfigure[]{\includegraphics[width=0.235\te